# 🔧 Jenkins — Ultra-Elaborate Mental Models

> **Every section answers four questions: WHY this exists, WHAT it is, HOW it works, WHEN to use it.**
> Real-world scenarios, ❌ before / ✅ after code, and *"Where this is seen in frameworks"* callouts.

---

**Topics**
1. Jenkins Architecture — The Controller-Agent Model
2. Declarative vs Scripted Pipelines
3. Jenkinsfile Deep Dive — Production-Grade Pipelines
4. Shared Libraries — DRY Pipelines at Scale
5. Multi-Branch Pipelines and Organization Folders
6. Plugin Ecosystem and Anti-Patterns
7. Jenkins vs GitHub Actions — The Architectural Decision
8. Real-World: Banks and Regulated Industries
9. The Jenkins Architect's Design Framework

---
## 1 · Jenkins Architecture — The Controller-Agent Model

### 🧠 Mental Model — *The Factory Floor*

> **Jenkins is a factory: the Controller (foreman) schedules work and assigns it to Agents (workers). Agents are the machines where code actually runs. The Controller should NEVER run build jobs — it is the brain, not the muscle. A Controller running builds is like a foreman doing all the welding — the factory doesn't scale.**

**WHY Jenkins exists:** Born in 2004 as Hudson (renamed Jenkins in 2011 after Oracle/Sun acquisition dispute). Before Jenkins, build automation was either manual, cron scripts, or expensive commercial tools (TeamCity, Bamboo). Jenkins democratized CI/CD with an open-source, extensible platform.

**WHY Jenkins is still dominant in 2024:** Despite GitHub Actions and GitLab CI, Jenkins dominates in:
- **Regulated industries** (banking, healthcare, aerospace) where pipelines must run on-premises
- **Large enterprises** with existing Jenkins infrastructure (migration cost is high)
- **Complex build graphs** where Groovy DSL provides more power than YAML
- **Multi-cloud** environments where one CI system must orchestrate across AWS, GCP, and on-prem

### Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────────┐
│                        Jenkins Controller                            │
│  ┌──────────────┐  ┌───────────────┐  ┌────────────────────────┐   │
│  │  Job Queue   │  │  Build Queue  │  │  Plugin Manager        │   │
│  │  Scheduler   │  │  History DB   │  │  (1,800+ plugins)      │   │
│  └──────────────┘  └───────────────┘  └────────────────────────┘   │
│         │                                                            │
│    JNLP / SSH / WebSocket connections to agents                     │
└─────────────────────────────────────────────────────────────────────┘
          │                    │                      │
    ┌─────▼──────┐      ┌──────▼──────┐       ┌──────▼──────┐
    │ Agent: VM  │      │ Agent: K8s  │       │ Agent: EC2  │
    │ Linux node │      │ Pod (ephemeral│     │ Spot instance│
    │ label:build│      │ label:docker)│     │ label:heavy  │
    └────────────┘      └─────────────┘       └─────────────┘
       Runs Java/        Runs Docker builds     Heavy compilation
       Python tests      on K8s cluster         (GPU ML models)
```

### Connection Protocols

| Protocol | How it works | Use case |
|---|---|---|
| **JNLP (Inbound)** | Agent initiates connection to Controller | Agents in private networks, firewalls blocking inbound |
| **SSH (Outbound)** | Controller SSH-es into agent | Traditional VMs where Controller can reach agents |
| **WebSocket** | Agent connects via WebSocket | Modern replacement for JNLP |
| **Kubernetes** | Controller creates ephemeral K8s pods | Elastic scaling — pods spun up per build, destroyed after |
| **EC2 Plugin** | Controller launches EC2 instances | Auto-scaling cloud agents |

### The Executor Model

```
Each agent has N executors (parallel build slots).
An executor is a thread that can run one pipeline stage at a time.

Agent: build-node-01 (4 executors)
  Executor 1: Running 'pytest unit/' for service-A
  Executor 2: Running 'npm test' for frontend
  Executor 3: Building Docker image for service-B
  Executor 4: (idle — available)

Rule of thumb: executors = CPU cores for CPU-bound jobs,
               executors = 2× cores for I/O-bound jobs (tests waiting on DB)
```

### 🌍 Real-World: Large Bank Jenkins Architecture
A major investment bank runs Jenkins with:
- 1 primary Controller (HA mode with 3 replicas)
- 200+ permanent agents on-premises (regulatory: code cannot leave the network)
- 500+ ephemeral K8s agents on a private OpenShift cluster
- 50,000+ builds per day
- Shared library with 200+ functions covering compliance, security scanning, deployment
- Every production deployment requires a cryptographically signed approval artifact

### ⚠️ Controller Anti-Patterns

```
❌ Running builds on the Controller
   Risk: Resource exhaustion → Controller crashes → ALL CI stops
   Fix: agent { label 'build' } — never agent none on the Controller

❌ Storing secrets in Jenkins Credentials without rotation
   Risk: Long-lived secrets = long blast radius if Jenkins is compromised
   Fix: Integrate with HashiCorp Vault for dynamic, short-lived credentials

❌ No Controller backup
   Risk: Controller disk failure = lose all job configurations and history
   Fix: Jenkins Configuration as Code (JCasC) + regular JENKINS_HOME backups

❌ Too many plugins
   Risk: Plugin conflicts, slow startup, security vulnerabilities
   Fix: Plugin inventory audit — remove unused plugins quarterly
```

In [ ]:
"""
Jenkins Controller-Agent Scheduler Simulator
============================================
Models how Jenkins assigns pipeline stages to agents based on labels.
This is the 'executor allocation' algorithm at the heart of Jenkins.
"""
from __future__ import annotations
import asyncio
import random
import time
from dataclasses import dataclass, field
from collections import deque
from typing import Optional, List, Set

In [ ]:
@dataclass
class Stage:
    name: str
    required_label: str    # which agent type this stage needs
    duration_s: float
    assigned_agent: Optional[str] = None
    start_time: Optional[float] = None
    end_time: Optional[float] = None


@dataclass
class Agent:
    name: str
    labels: Set[str]
    num_executors: int
    _busy_executors: int = 0

    @property
    def available_executors(self) -> int:
        return self.num_executors - self._busy_executors

    def can_run(self, label: str) -> bool:
        return label in self.labels and self.available_executors > 0

    def acquire(self) -> None:
        self._busy_executors += 1

    def release(self) -> None:
        self._busy_executors -= 1


class JenkinsScheduler:
    """Simplified Jenkins build queue and executor allocation."""

    def __init__(self, agents: List[Agent]):
        self.agents = agents
        self.queue: deque[Stage] = deque()
        self._t0 = time.monotonic()

    def _t(self) -> float:
        return time.monotonic() - self._t0

    def _find_agent(self, label: str) -> Optional[Agent]:
        """Jenkins picks the agent with most available executors (simplified)."""
        candidates = [a for a in self.agents if a.can_run(label)]
        return max(candidates, key=lambda a: a.available_executors, default=None)

    async def _execute_stage(self, stage: Stage, agent: Agent) -> None:
        agent.acquire()
        stage.assigned_agent = agent.name
        stage.start_time = self._t()
        print(f"  [{stage.start_time:4.1f}s] ▶ {stage.name:25s} → {agent.name}")
        await asyncio.sleep(stage.duration_s)
        stage.end_time = self._t()
        print(f"  [{stage.end_time:4.1f}s] ✓ {stage.name:25s} done (took {stage.duration_s:.1f}s)")
        agent.release()

    async def run_pipeline(self, stages: List[Stage]) -> None:
        tasks = []
        for stage in stages:
            agent = self._find_agent(stage.required_label)
            if not agent:
                print(f"  [{self._t():.1f}s] ⏳ {stage.name} queued (no '{stage.required_label}' agent available)")
                # Wait for an agent to free up
                while not (agent := self._find_agent(stage.required_label)):
                    await asyncio.sleep(0.1)
            tasks.append(asyncio.create_task(self._execute_stage(stage, agent)))
        await asyncio.gather(*tasks)


agents = [
    Agent("build-node-01", labels={"build", "python", "test"}, num_executors=3),
    Agent("build-node-02", labels={"build", "python", "test"}, num_executors=3),
    Agent("docker-node-01", labels={"docker", "build"}, num_executors=2),
    Agent("k8s-deploy-pod", labels={"deploy", "kubernetes"}, num_executors=1),
]

pipeline_stages = [
    Stage("Lint (ruff)",         required_label="python",  duration_s=1.0),
    Stage("Unit Tests",           required_label="test",    duration_s=2.5),
    Stage("Integration Tests",    required_label="test",    duration_s=4.0),
    Stage("Build Docker Image",   required_label="docker",  duration_s=3.0),
    Stage("Push to Registry",     required_label="docker",  duration_s=1.5),
    Stage("Deploy to K8s",        required_label="deploy",  duration_s=2.0),
]

print("=== Jenkins Pipeline Execution (agent-based scheduling) ===")
print(f"Agents: {[a.name for a in agents]}")
print()
scheduler = JenkinsScheduler(agents)
await scheduler.run_pipeline(pipeline_stages)

---
## 2 · Declarative vs Scripted Pipelines

### 🧠 Mental Model — *YAML-like Structure vs Full Groovy Power*

> **Declarative pipelines are to Jenkinsfiles what Terraform is to Bash scripts — a constrained, structured format that's easier to validate and understand. Scripted pipelines are full Groovy — maximum power, maximum complexity. Use Declarative by default. Switch to Scripted only when Declarative literally cannot express what you need.**

### Declarative Pipeline — The 80% Solution

```groovy
// ✅ DECLARATIVE PIPELINE — structured, validated, recommended
pipeline {
    agent { label 'python-build' }  // Never use 'any' in production
    
    options {
        timeout(time: 30, unit: 'MINUTES')  // ✅ never hang
        buildDiscarder(logRotator(numToKeepStr: '30'))  // ✅ disk management
        disableConcurrentBuilds()  // ✅ prevent race conditions on main
        retry(3)  // ✅ transient failures
    }

    environment {
        DOCKER_REGISTRY = 'registry.company.com'
        IMAGE_TAG = "${env.GIT_COMMIT[0..7]}"  // Short SHA as image tag
        // ✅ Secrets injected from Jenkins Credentials — never hardcoded
        SONAR_TOKEN = credentials('sonar-token-id')
    }

    stages {
        stage('Lint & Test') {
            parallel {
                stage('Lint') {
                    steps { sh 'ruff check .' }
                }
                stage('Unit Tests') {
                    steps {
                        sh 'pytest tests/unit/ --junitxml=reports/unit.xml'
                    }
                    post {
                        always {
                            junit 'reports/unit.xml'  // publish test results
                        }
                    }
                }
            }
        }
        
        stage('Build Image') {
            agent { label 'docker' }  // switch to docker-capable agent
            steps {
                sh "docker build -t ${DOCKER_REGISTRY}/myapp:${IMAGE_TAG} ."
                sh "docker push ${DOCKER_REGISTRY}/myapp:${IMAGE_TAG}"
            }
        }
        
        stage('Deploy') {
            when {
                branch 'main'  // only deploy from main
            }
            input {
                message "Deploy to production?"
                ok "Deploy"
                submitter "lead-engineers"  // only certain users can approve
            }
            steps {
                sh "kubectl set image deployment/myapp app=${DOCKER_REGISTRY}/myapp:${IMAGE_TAG}"
            }
        }
    }
    
    post {
        failure {
            slackSend(color: 'danger', message: "BUILD FAILED: ${env.JOB_NAME} #${env.BUILD_NUMBER}")
        }
        success {
            slackSend(color: 'good', message: "Deployed ${IMAGE_TAG} to production")
        }
        always {
            cleanWs()  // ✅ clean workspace to prevent disk fill
        }
    }
}
```

### Scripted Pipeline — When You Need the Power

```groovy
// ✅ SCRIPTED PIPELINE — used when Declarative can't express the logic
// Example: dynamic parallel stages generated from configuration
node('build') {
    def services = ['payment-service', 'user-service', 'notification-service']
    
    // Dynamically create parallel stages — impossible in Declarative
    def parallelStages = services.collectEntries { service ->
        [
            "Test: ${service}": {
                dir(service) {
                    sh "pytest tests/ --junitxml=../reports/${service}.xml"
                }
            }
        ]
    }
    
    stage('Parallel Service Tests') {
        parallel parallelStages  // all services tested simultaneously
    }
}
```

### Decision Matrix

| Requirement | Use Declarative | Use Scripted |
|---|---|---|
| Standard CI pipeline | ✅ | |
| Dynamic parallel stages from data | | ✅ |
| Complex error handling / try-catch | | ✅ |
| Validation by Jenkins UI | ✅ | |
| Readability for all team members | ✅ | |
| Calling shared library functions | ✅ both work | ✅ both work |

In [ ]:
"""
Jenkinsfile Generator
=====================
Programmatically generates production-grade Jenkinsfiles.
This is the kind of tooling a DevOps team builds to ensure
all 50 services follow the same pipeline standards.
"""
from __future__ import annotations
from dataclasses import dataclass, field
from typing import List, Optional
import textwrap


@dataclass
class ServiceConfig:
    name: str
    language: str              # python, java, nodejs
    has_docker: bool = True
    deploy_to_k8s: bool = True
    requires_manual_approval: bool = False
    notify_slack_channel: str = "#ci-alerts"
    test_commands: List[str] = field(default_factory=list)


class JenkinsfileGenerator:
    """Generates standardized Declarative Jenkinsfiles from service config."""

    TEST_COMMANDS = {
        "python": "pytest tests/ -v --junitxml=reports/test-results.xml --cov=src --cov-report=xml",
        "java":   "./mvnw test -Dsurefire.reportsDirectory=reports/",
        "nodejs": "npm test -- --reporter=junit --reporter-options output=reports/test-results.xml",
    }

    LINT_COMMANDS = {
        "python": "ruff check . && ruff format --check .",
        "java":   "./mvnw checkstyle:check",
        "nodejs": "npm run lint",
    }

    def generate(self, config: ServiceConfig) -> str:
        test_cmd  = self.TEST_COMMANDS.get(config.language, "echo 'no test command'")
        lint_cmd  = self.LINT_COMMANDS.get(config.language, "echo 'no lint command'")
        manual_approval = ""
        if config.requires_manual_approval:
            manual_approval = """
            input {
                message 'Approve production deployment?'
                ok 'Deploy to Production'
                submitter 'lead-engineers,platform-team'
            }"""

        return textwrap.dedent(f"""
            // Generated Jenkinsfile for: {config.name}
            // Language: {config.language} | Docker: {config.has_docker} | K8s: {config.deploy_to_k8s}
            // DO NOT EDIT — managed by platform team via JenkinsfileGenerator

            @Library('company-shared-library@main') _

            pipeline {{
                agent {{ label '{config.language}-build' }}

                options {{
                    timeout(time: 30, unit: 'MINUTES')
                    buildDiscarder(logRotator(numToKeepStr: '30'))
                    disableConcurrentBuilds(abortPrevious: true)
                }}

                environment {{
                    SERVICE_NAME = '{config.name}'
                    IMAGE_TAG    = "${{env.GIT_COMMIT[0..7]}}"
                    REGISTRY     = credentials('docker-registry-url')
                }}

                stages {{
                    stage('Checkout') {{
                        steps {{ checkout scm }}
                    }}

                    stage('Quality Gate') {{
                        parallel {{
                            stage('Lint') {{
                                steps {{ sh '{lint_cmd}' }}
                            }}
                            stage('Unit Tests') {{
                                steps {{
                                    sh '{test_cmd}'
                                }}
                                post {{
                                    always {{
                                        junit 'reports/test-results.xml'
                                        publishCoverage adapters: [istanbulCoberturaAdapter('reports/coverage.xml')]
                                    }}
                                }}
                            }}
                            stage('Security Scan') {{
                                steps {{ sh 'trivy fs --exit-code 1 --severity CRITICAL .' }}
                            }}
                        }}
                    }}
                    {self._docker_stages(config) if config.has_docker else ''}
                    {self._deploy_stage(config, manual_approval) if config.deploy_to_k8s else ''}
                }}

                post {{
                    failure   {{ notifySlack('{config.notify_slack_channel}', 'FAILED', env.BUILD_URL) }}
                    success   {{ notifySlack('{config.notify_slack_channel}', 'SUCCESS', env.BUILD_URL) }}
                    always    {{ cleanWs() }}
                }}
            }}
        """).strip()

    def _docker_stages(self, config: ServiceConfig) -> str:
        return """
                    stage('Build & Push Image') {
                        agent { label 'docker' }
                        steps {
                            sh """docker buildx build \\
                                --platform linux/amd64,linux/arm64 \\
                                --cache-from ${REGISTRY}/${SERVICE_NAME}:cache \\
                                --cache-to type=inline \\
                                --tag ${REGISTRY}/${SERVICE_NAME}:${IMAGE_TAG} \\
                                --push ."""
                        }
                    }"""

    def _deploy_stage(self, config: ServiceConfig, manual_approval: str) -> str:
        return f"""
                    stage('Deploy to Production') {{
                        when {{ branch 'main' }}
                        {manual_approval}
                        steps {{
                            sh """kubectl set image deployment/{config.name} \\
                                {config.name}=${{REGISTRY}}/{config.name}:${{IMAGE_TAG}} \\
                                --record"""
                            sh 'kubectl rollout status deployment/{config.name} --timeout=5m'
                        }}
                    }}"""


gen = JenkinsfileGenerator()
payment_service = ServiceConfig(
    name="payment-service",
    language="python",
    requires_manual_approval=True,  # Payments need human gate
    notify_slack_channel="#payments-deploys",
)

jenkinsfile = gen.generate(payment_service)
print(jenkinsfile)

---
## 3 · Shared Libraries — DRY Pipelines at Scale

### 🧠 Mental Model — *The Platform Team's Gift to Product Teams*

> **A Jenkins Shared Library is a Groovy code repository that all pipelines in an organization can import. It is the same concept as a Python library — reusable functions that abstract complex, organization-specific logic. The platform team maintains the library; product teams consume it. This is how one platform team supports 100 product teams without becoming a bottleneck.**

**WHY shared libraries exist:** Without them:
- 50 microservices → 50 duplicate pipeline definitions
- Security fix requires updating 50 files
- Standard compliance checks drift as teams modify their pipelines
- Onboarding a new service requires copy-pasting a Jenkinsfile and hoping nothing important was missed

**WHAT the library provides:**
1. **Vars** — Global variables/functions callable from any Jenkinsfile
2. **Src** — Full Groovy classes with tests
3. **Resources** — Static config files, scripts, templates

### Shared Library Directory Structure

```
company-jenkins-library/
├── vars/
│   ├── buildDockerImage.groovy   # called as: buildDockerImage('myapp')
│   ├── deployToK8s.groovy        # called as: deployToK8s(env: 'staging')
│   ├── notifySlack.groovy        # called as: notifySlack('#deploys', 'PASS')
│   ├── runSecurityScan.groovy    # called as: runSecurityScan()
│   └── pythonPipeline.groovy     # entire pipeline in one call!
├── src/
│   └── com/company/ci/
│       ├── DockerUtils.groovy    # Groovy class for Docker operations
│       └── SlackNotifier.groovy  # Groovy class for Slack integration
├── resources/
│   ├── sonar-project.properties.template
│   └── default-jvm-opts.txt
└── test/
    └── groovy/
        └── BuildDockerImageSpec.groovy  # Spock tests for the library
```

### The Ultimate Pattern: Pipeline-in-a-Function

```groovy
// vars/pythonMicroservicePipeline.groovy
// A complete, opinionated pipeline for Python services
// Product team's Jenkinsfile becomes:
//
//   @Library('company-lib@main') _
//   pythonMicroservicePipeline(serviceName: 'payment-service', requiresApproval: true)

def call(Map config = [:]) {
    def serviceName     = config.serviceName ?: error('serviceName required')
    def requiresApproval = config.get('requiresApproval', false)
    def deployBranch    = config.get('deployBranch', 'main')
    
    pipeline {
        agent { label 'python-build' }
        
        options {
            timeout(time: 30, unit: 'MINUTES')
            buildDiscarder(logRotator(numToKeepStr: '30'))
        }
        
        stages {
            stage('Quality Gate') {
                parallel {
                    stage('Lint')          { steps { runLint() } }
                    stage('Tests')         { steps { runPytests(serviceName) } }
                    stage('Security')      { steps { runSecurityScan() } }
                }
            }
            stage('Build') {
                steps { buildDockerImage(serviceName) }
            }
            stage('Deploy') {
                when { branch deployBranch }
                steps { deployToK8s(serviceName: serviceName, requiresApproval: requiresApproval) }
            }
        }
        
        post {
            failure { notifySlack("#${serviceName}-alerts", 'FAILED') }
            always  { cleanWs() }
        }
    }
}
```

**Result:** A product team's entire Jenkinsfile is 3 lines:
```groovy
@Library('company-lib@main') _
pythonMicroservicePipeline(serviceName: 'payment-service', requiresApproval: true)
```

**Where this is seen:** Netflix OSS Spinnaker pipeline templates, Spotify's Backstage CI templates, and large enterprise Jenkins setups universally converge on this pattern.

---
## 4 · Jenkins vs GitHub Actions — The Architectural Decision

### 🧠 Mental Model — *Fit to Context, Not Fashion*

> **Choosing Jenkins vs GitHub Actions is not a technology preference — it is an architectural decision driven by compliance requirements, existing infrastructure, and organizational constraints. A senior architect evaluates both and selects based on context.**

### Comprehensive Comparison

| Dimension | Jenkins | GitHub Actions |
|---|---|---|
| **Hosting** | Self-hosted (you manage the server) | Cloud (GitHub manages) |
| **Build execution** | Self-hosted agents | GitHub-hosted runners or self-hosted |
| **Pipeline language** | Groovy DSL (Declarative or Scripted) | YAML |
| **Expressiveness** | Higher (full programming language) | Medium (limited YAML constructs) |
| **Marketplace** | 1,800+ plugins | 20,000+ actions |
| **Compliance** | ✅ Full on-premises, air-gapped possible | Requires self-hosted for on-prem |
| **Maintenance burden** | High (upgrades, plugins, security patches) | Low (managed by GitHub) |
| **Cost** | Infrastructure + engineer time | Per-minute pricing for private repos |
| **Scalability** | Manual agent scaling (or K8s plugin) | Automatic (GitHub-hosted) |
| **Integration with Git** | Via webhook plugin | Native — same platform |
| **Secret management** | Jenkins Credentials + Vault plugin | GitHub Secrets + OIDC |
| **Audit trail** | Jenkins audit log plugin | GitHub audit log |

### When to Choose Jenkins

```
Choose Jenkins when:

1. COMPLIANCE REQUIREMENT: Code and builds cannot leave your network
   (Banking, Defense, Healthcare with strict data residency requirements)
   
2. EXISTING INVESTMENT: $2M of Jenkins infrastructure + 50 Shared Libraries
   Migration cost exceeds benefit for most orgs (unless Jenkins causes pain)
   
3. COMPLEX ORCHESTRATION: Multi-repo, multi-language, multi-cloud builds
   where Groovy provides flexibility YAML cannot
   
4. HETEROGENEOUS INFRASTRUCTURE: Builds on Windows, Linux, mainframe, AIX
   Jenkins plugins exist for almost every OS and platform
   
5. ENTERPRISE FEATURES: Role-based access, AD/LDAP integration,
   audit compliance, artifact management — all available as plugins
```

### When to Choose GitHub Actions

```
Choose GitHub Actions when:

1. GREENFIELD: No existing CI investment — Actions is dramatically faster to start

2. STARTUP/SMB: No platform team to maintain Jenkins — managed CI is lower TCO

3. OPEN SOURCE: Public repos get unlimited free minutes on GitHub-hosted runners

4. GITHUB-NATIVE: Deep integration with PR checks, deployments, environments

5. SIMPLICITY: A YAML pipeline that any engineer can read is worth more
   than a Groovy pipeline only the CI team understands
```

### 🌍 Real-World: The Migration Story
A fintech company with 150 engineers ran Jenkins for 5 years. Pain points:
- 2 engineers full-time maintaining Jenkins infrastructure
- Plugin updates breaking pipelines every quarter
- 45-minute pipelines due to a lack of parallelism

They migrated to GitHub Actions over 6 months:
- Pipeline time dropped from 45 to 9 minutes (parallelism + caching)
- 0 engineers needed for CI infrastructure maintenance
- Onboarding a new service went from 1 week to 2 hours

**BUT:** They kept Jenkins for compliance-sensitive services where builds had to stay on-prem.
This hybrid model (GitHub Actions for most, Jenkins for regulated) is increasingly common.

---

## 5 · The Jenkins Architect's Design Framework

### Before Building Any Jenkins Pipeline, Answer:

```
1. CONTROLLER SIZING
   - How many concurrent builds? (Size the Controller accordingly)
   - Is HA required? (Use JNLP agents + persistent storage + active-passive Controller)

2. AGENT STRATEGY  
   - Static agents (VMs): predictable cost, warm — for frequent, fast builds
   - K8s ephemeral pods: elastic, zero idle cost — for variable workloads
   - EC2 on-demand: elastic, slower startup — for cloud-native teams

3. PIPELINE STRATEGY
   - Shared library or per-service Jenkinsfile?
   - Declarative or Scripted?
   - Multi-branch or single-branch?

4. SECURITY
   - Vault integration for secrets (not just Jenkins Credentials)
   - Role-Based Access Control (RBAC) plugin
   - Script Security plugin (prevent arbitrary Groovy execution)
   - Audit trail plugin for compliance

5. OBSERVABILITY
   - Prometheus metrics via prometheus plugin
   - Build status dashboard (Blue Ocean or custom)
   - Alert on: build failure rate, queue time, agent offline events
```